## Demo Diffusion models generative AI

Author: alberto.suarez@uam.es
Date: 2025-03-02

In [10]:
%load_ext autoreload
%autoreload 2

import numpy as np

from functools import partial 

import torch
from torch.utils.data import (
    DataLoader,
    Dataset,
    Subset,
)
from torchvision import datasets
from torchvision.transforms import ToTensor
from torchvision.transforms import functional

import diffusion_process as dfp

from diffusion_utilities import (
    plot_image_grid,
    plot_image_evolution,
    animation_images,
)


n_threads = torch.get_num_threads()
print('Number of threads: {:d}'.format(n_threads))

device ='cpu' 


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Number of threads: 6


In [11]:
batch_size = 128

data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)
data_train = data
data_loader = DataLoader(
    data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

###  Diffusion process

In [12]:

# 1. alpha_bar (cosine schedule)
def alpha_bar(t):
    s = 0.008
    s_tensor = torch.tensor(s, device=t.device, dtype=t.dtype)

    a_bar = (
        torch.cos((t + s_tensor) / (1 + s_tensor) * torch.pi / 2) ** 2
        / torch.cos((s_tensor / (1 + s_tensor)) * torch.pi / 2) ** 2
    )

    return torch.clamp(a_bar, min=1.0e-12, max=1.0)



# 2. beta(t)
def beta_t(t):
    s = 0.008
    beta = (torch.pi / (1 + s)) * torch.tan(
        (t + s) / (1 + s) * torch.pi / 2
    )
    return torch.clamp(beta, min=1.0e-12, max=50.0)



# 3. drift

def drift(x_t, t):
    beta = beta_t(t)
    return -0.5 * beta[:, None, None, None] * x_t


# -----------------------------
# 4. diffusion
# -----------------------------
def diffusion(t):
    beta = beta_t(t)
    return torch.sqrt(beta)



# 5. mu_t

def mu_t(x_0, t):
    a_bar = alpha_bar(t)
    return torch.sqrt(a_bar)[:, None, None, None] * x_0



# 6. sigma_t

def sigma_t(t):
    a_bar = alpha_bar(t)
    return torch.sqrt(torch.clamp(1.0 - a_bar, min=1.0e-12))



# 7. construir proceso

diffusion_process = dfp.GaussianDiffussionProcess(
    drift,
    diffusion,
    mu_t,
    sigma_t
)

In [13]:
# Define the score model

from score_model import ScoreNet

score_model = torch.nn.DataParallel(
    ScoreNet(
        marginal_prob_std=sigma_t
    )
)
score_model = score_model.to(device)

In [14]:
# Train model
from torch.optim import Adam #optimizador calcula los pesos de la red
import torchvision.transforms as transforms 
from tqdm.notebook import trange #barra de progreso

batch_size = 32 #num. imagenes procesadas en cada iteración

data_loader = DataLoader( #dataset de 3's
    data_train, 
    batch_size=batch_size, 
    shuffle=True, #mezcla datos en cada epoca
    num_workers=n_threads, #paralelizar el proceso
)

#  [TO DO: Comment each line of code]


learning_rate = 1.0e-3
optimizer = Adam(score_model.parameters(), lr=learning_rate) #optimizador para actualizar 
#los parámetros del modelo

n_epochs =  35
tqdm_epoch = trange(n_epochs) #barra de procesos

for epoch in tqdm_epoch:
    avg_loss = 0.0 #inicializas para calcular la pérdida media
    num_items = 0
    for x, y in data_loader:
        x = x.to(device)    
        loss = diffusion_process.loss_function(score_model, x) #función de pérdida
        optimizer.zero_grad() #eliminas gradientes previos
        loss.backward()    
        optimizer.step()
        avg_loss += loss.item() * x.shape[0]
        num_items += x.shape[0]
        
    tqdm_epoch.set_description('Average Loss: {:5f}'.format(avg_loss / num_items))

torch.save(score_model.state_dict(), 'vp_cosine.pth')

  0%|          | 0/35 [00:00<?, ?it/s]

importante lo de T= 1- e